[//]: # (cr:doc name='chapter_c03_approval_gate' id=3e5b050a)
# Chapter c03: Approval Gate (Causal Track)

Compares each new draft archetype to its prior `active` row by cosine similarity of the SHAP-space centroid vectors. Auto-promotes when `cos_sim ≥ STABILITY_THRESHOLD`; otherwise leaves the row as `pending_review` and prints it on the manual queue at the bottom.

Cascades the promotion to `eligibility_policy` via `arrays_overlap`.


In [ ]:
# @cr:code name='init_progress' id=83682b54
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("c03_approval_gate.ipynb")
# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


[//]: # (cr:doc name='c03_configuration' id=2e299dde)
## Configuration

The cell below is the only place you should need to edit. Every value here is read by the approval cells — nothing is hardcoded inside the algorithm.

- **`STABILITY_THRESHOLD`** — cosine similarity above which a re-derived archetype is auto-promoted to `active` without manual review. 0.95 is the recommended starting value; tune after observing 2-3 retrains. Lower the threshold if too many obviously-stable archetypes are tripping the manual gate; raise it if drifted clusters are slipping through without review.
- **`FORCE_APPROVE`** — flip every pending row to `active`. Use after a manual review.


In [ ]:
# @cr:config name='configuration' id=2d889a01
STABILITY_THRESHOLD = 0.95

FORCE_APPROVE = False

# === Optional engagement overrides — leave None for auto-detection ====
# RunNamespace.resolve() honours these first, then falls back to
# discovery tiers (CR_RUN_ID env, project pointer .cr_active_run.json,
# experiments_root/runs/.active_run_id sentinel, latest-by-mtime).
# Pin them when several runs share an experiments dir and auto-detection
# picked the wrong one.
ENGAGEMENT_RUN_ID = None
ENGAGEMENT_EXPERIMENTS_DIR = None
MODEL_URI_OVERRIDE = None  # if None, looked up from MLflow @production alias


[//]: # (cr:doc name='c03_approval_gate_setup' id=2ee514a7)
## 3.0 Setup

Resolves catalog / schema / model identifiers from `ScoringConfig` (reads the persisted Databricks init JSON on Databricks, or the local pipeline's `best_model_meta.json` for local runs). The composite-name-qualified gold features table name is derived here so the algorithmic cells stay free of path-construction logic.


In [ ]:
# @cr:code name='setup_and_resolve_model' id=5befb4ad
from customer_retention.core.compat.detection import get_spark_session, is_databricks
from customer_retention.core.config import get_playbooks_dir
from customer_retention.stages.scoring import resolve_scoring_context

spark = get_spark_session()
PLAYBOOKS_DIR = get_playbooks_dir()

# Auto-detect the active run / model via RunNamespace's file-tracked
# discovery tiers (CR_RUN_ID env, project pointer .cr_active_run.json,
# experiments_root/runs/.active_run_id sentinel, latest-by-mtime). On
# Databricks, the model URI is resolved from MLflow's @production alias
# for the registered model named in training_metadata.json. Operator
# overrides from the configuration cell above (ENGAGEMENT_RUN_ID /
# ENGAGEMENT_EXPERIMENTS_DIR / MODEL_URI_OVERRIDE) short-circuit each
# resolution tier when set; default None → auto-detect.
_resolved = resolve_scoring_context(
    run_id=globals().get("ENGAGEMENT_RUN_ID"),
    experiments_dir=globals().get("ENGAGEMENT_EXPERIMENTS_DIR"),
    model_uri=globals().get("MODEL_URI_OVERRIDE"),
)
scoring_config = _resolved.scoring_config
_namespace = _resolved.namespace
_ns_source = _resolved.source
CATALOG = scoring_config.catalog if is_databricks() else "local"
SCHEMA = scoring_config.schema if is_databricks() else "local"
MODEL_NAME = _resolved.model_name
MODEL_VERSION = _resolved.model_version
MODEL_URI = _resolved.model_uri

COMPOSITE_NAME = scoring_config.composite_name
GOLD_FEATURES_FQN = (
    f"{CATALOG}.{SCHEMA}.gold_features_{COMPOSITE_NAME}"
    if COMPOSITE_NAME
    else f"{CATALOG}.{SCHEMA}.gold_features"
)

ARCHETYPE_CATALOG_FQN = f"{CATALOG}.{SCHEMA}.archetype_catalog"
ELIGIBILITY_POLICY_FQN = f"{CATALOG}.{SCHEMA}.eligibility_policy"
DECISION_POLICY_FQN = f"{CATALOG}.{SCHEMA}.decision_policy"
ELIGIBILITY_SNAPSHOT_FQN = f"{CATALOG}.{SCHEMA}.eligibility_snapshot"
PREDICTIONS_FQN = f"{CATALOG}.{SCHEMA}.predictions"
TOP_SHAP_DRIVERS_FQN = f"{CATALOG}.{SCHEMA}.top_shap_drivers"

print(f"Resolved playbooks_dir: {PLAYBOOKS_DIR}")
print(f"Active run namespace:   {_namespace.run_id if _namespace else '(none)'}")
print(f"Run source:             {_ns_source}")
print(f"Catalog/schema:         {CATALOG}.{SCHEMA}")
print(f"Composite name:         {COMPOSITE_NAME or '(unset)'}")
print(f"Gold features table:    {GOLD_FEATURES_FQN}")
print(f"Model URI:              {MODEL_URI or '(local)'}")
print(f"Model version:          {MODEL_VERSION}")


[//]: # (cr:doc name='c03_resolve_run_section' id=f55caf19)
## 3.1 Resolve Latest Pending Derivation Run


In [ ]:
# @cr:code name='resolve_derivation_run_id' id=50e458f4
from customer_retention.stages.causal import expire_stale_pending

derivation_run_id = None
if spark is not None and spark.catalog.tableExists(ARCHETYPE_CATALOG_FQN):
    row = spark.sql(
        f"SELECT derivation_run_id FROM {ARCHETYPE_CATALOG_FQN} "
        "WHERE status = 'pending_review' AND model_name = ? AND model_version = ? "
        "ORDER BY derivation_run_id DESC LIMIT 1",
        args=[MODEL_NAME, MODEL_VERSION],
    ).head()
    derivation_run_id = row["derivation_run_id"] if row else None

if derivation_run_id is None:
    print("No pending_review derivations found for the current model version.")
else:
    expired = expire_stale_pending(
        spark, ARCHETYPE_CATALOG_FQN, ELIGIBILITY_POLICY_FQN,
        MODEL_NAME, MODEL_VERSION, derivation_run_id,
    )
    if expired:
        print(f"Expired {expired} stale pending_review rows from prior derivation runs")
    print(f"Resolved derivation_run_id: {derivation_run_id}")


[//]: # (cr:doc name='c03_run_gate_section' id=6cb63243)
## 3.2 Run Approval Gate


In [ ]:
# @cr:code name='approval_gate' id=ab8926d6
from customer_retention.stages.causal import auto_promote_stable

gate_result = None
if derivation_run_id is None:
    print("SKIPPED: nothing pending to approve")
else:
    gate_result = auto_promote_stable(
        spark=spark,
        archetype_table_fqn=ARCHETYPE_CATALOG_FQN,
        policy_table_fqn=ELIGIBILITY_POLICY_FQN,
        derivation_run_id=derivation_run_id,
        threshold=STABILITY_THRESHOLD,
        force=FORCE_APPROVE,
    )
    print(gate_result.summary())


[//]: # (cr:doc name='c03_print_pending_section' id=ff721880)
## 3.3 Print Pending Review Queue


In [ ]:
# @cr:code name='print_pending_queue' id=917be0ac
from customer_retention.stages.causal import list_pending_review

if derivation_run_id is None:
    print("(no pending queue — derivation_run_id not resolved)")
else:
    pending = list_pending_review(spark, ARCHETYPE_CATALOG_FQN, derivation_run_id)
    if not pending:
        print("All archetypes auto-promoted. Pending review queue is empty.")
    else:
        print(
            "Archetype pending-review queue (re-run this notebook with FORCE_APPROVE=True "
            "after manual review):"
        )
        for row in pending:
            print(
                f"  - {row['archetype_id']} v{row['archetype_version']} "
                f"name={row['name']!r} stability={row['stability_vs_prior_version']}"
            )

# Transparency: also show every pending eligibility_policy row with its fit tier
# and score. The approval gate auto-promotes only 'auto'-tier rows; 'review'
# and 'catch_all' rows stay pending for human decision.
if derivation_run_id is not None and spark.catalog.tableExists(ELIGIBILITY_POLICY_FQN):
    policy_rows = spark.sql(
        f"SELECT playbook_id, archetype_ids, fit_tier, fit_score, rationale "
        f"FROM {ELIGIBILITY_POLICY_FQN} "
        "WHERE derivation_run_id = ? AND status = 'pending_review' "
        "ORDER BY fit_tier, fit_score DESC",
        args=[derivation_run_id],
    ).collect()
    if policy_rows:
        print("\nEligibility policy pending-review queue:")
        for r in policy_rows:
            arch = (r["archetype_ids"] or ["?"])[0]
            score = r["fit_score"]
            score_str = f"{float(score):.2f}" if score is not None else "n/a"
            rationale = (r["rationale"] or "")[:140]
            print(
                f"  - [{r['fit_tier']}] playbook={r['playbook_id']} "
                f"archetype_version={arch} fit_score={score_str} | {rationale}"
            )
    else:
        print("\nEligibility policy pending-review queue is empty.")


In [ ]:
# @cr:code name='release_stage_memory' id=47a76847
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
